In [3]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
import pandas as pd
# Assume:
# X = your dataframe (pandas)
# y = your target
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

y = train['INDICATED_DAMAGE']
X = train.drop('INDICATED_DAMAGE', axis=1)

# Identify categorical columns (by name or index)
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X[cat_cols] = X[cat_cols].fillna('missing').astype(str)
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
    
# Create CatBoost Pools (important!)
train_pool = Pool(X_train, y_train, cat_features=cat_cols)
val_pool = Pool(X_val, y_val, cat_features=cat_cols)

# Model
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',          # change if needed
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100
)

# Train
model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True
    
)

# Predictions
preds = model.predict(val_pool)
probs = model.predict_proba(val_pool)[:, 1]

# Save IDs first
test_ids = test['INDEX_NR']

# Match training features
X_test = test[X.columns].copy()

# Apply same categorical preprocessing
X_test[cat_cols] = X_test[cat_cols].fillna('missing').astype(str)

# Predict best class labels: 0 or 1
test_preds = model.predict(X_test).astype(int).flatten()

submission = pd.DataFrame({
    'INDEX_NR': test_ids,
    'INDICATED_DAMAGE': test_preds
})

submission.to_csv('submission.csv', index=False)

print(submission.head())

C:\Users\thanh\AppData\Local\Temp\ipykernel_29528\3331894371.py:7: DtypeWarning: Columns (8,9,20) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('train.csv')
C:\Users\thanh\AppData\Local\Temp\ipykernel_29528\3331894371.py:8: DtypeWarning: Columns (8,9,20,37) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv('test.csv')


0:	test: 0.8502989	best: 0.8502989 (0)	total: 2.03s	remaining: 33m 45s
100:	test: 0.9134309	best: 0.9134309 (100)	total: 3m 26s	remaining: 30m 39s
200:	test: 0.9188362	best: 0.9188362 (200)	total: 6m 35s	remaining: 26m 10s
300:	test: 0.9210043	best: 0.9210043 (300)	total: 9m 24s	remaining: 21m 51s
400:	test: 0.9222792	best: 0.9222792 (400)	total: 12m 33s	remaining: 18m 45s
500:	test: 0.9229380	best: 0.9229380 (500)	total: 15m 31s	remaining: 15m 27s
600:	test: 0.9233788	best: 0.9233788 (600)	total: 18m 31s	remaining: 12m 17s
700:	test: 0.9238440	best: 0.9238440 (699)	total: 21m 29s	remaining: 9m 10s
800:	test: 0.9242364	best: 0.9242364 (800)	total: 24m 24s	remaining: 6m 3s
900:	test: 0.9245526	best: 0.9245526 (900)	total: 27m 28s	remaining: 3m 1s
999:	test: 0.9248283	best: 0.9248283 (999)	total: 30m 27s	remaining: 0us

bestTest = 0.9248283334
bestIteration = 999

   INDEX_NR  INDICATED_DAMAGE
0   9000000                 0
1   9000001                 0
2   9000002                 0
3   9